In [1]:
import os
import sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
sys.path.append(os.path.expanduser("/home/guyb/tuner_knowledge"))


In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
import torch
from src.shared_prompt import SYSTEM_PROMPT
from transformers import DataCollatorForSeq2Seq
from transformers import TrainingArguments, Trainer
from src.triviaQA_load import take_first_n, stream_triviaqa_rc
from datasets import Dataset

In [3]:
dataset=stream_triviaqa_rc(batch_size=10)
raw_ds = take_first_n(dataset, 10)

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

In [4]:
def format_prompt(example):
    answer = example["answer"]["normalized_value"]
    prompt = SYSTEM_PROMPT.replace("{question}", example["question"])
    full = prompt + " " + answer
    return {
        "text": full,
        "input": prompt,
        "label": answer
    }

In [5]:
# Apply format_prompt to all examples and flatten in one step
formatted_examples = [
    format_prompt(example)
    for batch in raw_ds
    for example in batch
]
dataset = Dataset.from_list(formatted_examples)


In [6]:
model_id = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token 

def tokenize_fn(example):
    full_text = example["text"]
    prompt_text = example["input"]

    # Tokenize
    full_tokens = tokenizer(full_text, truncation=True)
    prompt_tokens = tokenizer(prompt_text, truncation=True)

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]

    # Build labels
    labels = input_ids.copy()
    prompt_len = len(prompt_tokens["input_ids"])
    labels[:prompt_len] = [-100] * min(prompt_len, len(labels))

    # 🔑 Pad labels to match input length (required!)
    while len(labels) < len(input_ids):
        labels.append(-100)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


In [7]:
tokenized_dataset = dataset.map(tokenize_fn, remove_columns=["label","text", "input"])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [8]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_id,
    padding=True,
    label_pad_token_id=-100
)

In [12]:
# Correct batching
batch = [tokenized_dataset[i] for i in range(50)]

# Pad dynamically
collated = data_collator(batch)

# Sanity check
for k, v in collated.items():
    print(k, v.shape)


input_ids torch.Size([50, 57])
attention_mask torch.Size([50, 57])
labels torch.Size([50, 57])


In [10]:
stop

NameError: name 'stop' is not defined

In [13]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda",
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 2,293,760 || all params: 3,215,043,584 || trainable%: 0.0713


In [14]:
training_args = TrainingArguments(
    output_dir="llama3_lora_fp16",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="no",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

[2025-08-03 13:22:39,943] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/opt/anaconda3/envs/guyb_fml/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/opt/anaconda3/envs/guyb_fml/compiler_compat/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


[2025-08-03 13:22:40,916] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [15]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=7, training_loss=3.7302148001534596, metrics={'train_runtime': 4.0541, 'train_samples_per_second': 24.666, 'train_steps_per_second': 1.727, 'total_flos': 85849924337664.0, 'train_loss': 3.7302148001534596, 'epoch': 1.0})

In [ ]:
trainer.save_model("models/llama3_lora_adapter")
tokenizer.save_pretrained("models/llama3_lora_adapter")

In [4]:
import os
import sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
sys.path.append(os.path.expanduser("/home/guyb/tuner_knowledge"))

In [9]:
import json
from collections import Counter

jsonl_path = "/home/guyb/tuner_knowledge/src/results/meta-llama_Llama-3.2-3B_scores.jsonl"  # replace with your actual path

with open(jsonl_path, "r", encoding="utf-8") as f:
    ids = [json.loads(line)["id"] for line in f if line.strip()]

counts = Counter(ids)

duplicates = {k: v for k, v in counts.items() if v > 1}
if duplicates:
    print("❌ Duplicate IDs found:")
    for k, v in duplicates.items():
        print(f"  ID '{k}' appears {v} times")
else:
    print("✅ All IDs are unique.")


❌ Duplicate IDs found:
  ID 'tc_3' appears 2 times
  ID 'tc_8' appears 2 times
  ID 'tc_9' appears 2 times
  ID 'tc_10' appears 2 times
  ID 'tc_11' appears 2 times
  ID 'tc_15' appears 2 times
  ID 'tc_16' appears 2 times
  ID 'tc_17' appears 2 times
  ID 'tc_21' appears 2 times
  ID 'tc_22' appears 2 times
  ID 'tc_23' appears 2 times
  ID 'tc_24' appears 2 times
  ID 'tc_25' appears 2 times
  ID 'tc_26' appears 2 times
  ID 'tc_27' appears 2 times
  ID 'tc_30' appears 2 times
  ID 'tc_31' appears 2 times
  ID 'tc_32' appears 2 times
  ID 'tc_38' appears 2 times
  ID 'tc_39' appears 2 times
  ID 'tc_41' appears 2 times
  ID 'tc_43' appears 2 times
  ID 'tc_45' appears 2 times
  ID 'tc_46' appears 2 times
  ID 'tc_47' appears 2 times
  ID 'tc_48' appears 2 times
  ID 'tc_50' appears 2 times
  ID 'tc_52' appears 2 times
  ID 'tc_53' appears 2 times
  ID 'tc_55' appears 2 times
  ID 'tc_58' appears 2 times
  ID 'tc_60' appears 2 times
  ID 'tc_61' appears 2 times
  ID 'tc_63' appears 2 

In [10]:
import json
from collections import defaultdict, Counter

jsonl_path = "/home/guyb/tuner_knowledge/src/results/meta-llama_Llama-3.2-3B_scores.jsonl"

# Load all lines with parsed JSON and their line numbers
lines = []
with open(jsonl_path, "r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        if line.strip():
            obj = json.loads(line)
            lines.append((obj["id"], idx + 1, obj))  # store id, line number, and object

# Count IDs
ids = [id_ for id_, _, _ in lines]
counts = Counter(ids)

# Collect duplicate examples
duplicates = {k: v for k, v in counts.items() if v > 1}

if not duplicates:
    print("✅ All IDs are unique.")
else:
    print("❌ Duplicate IDs found:")
    for dup_id, count in list(duplicates.items())[:3]:  # show only first 3 duplicates
        print(f"\n🔁 ID '{dup_id}' appears {count} times. Sample lines:")
        for id_, line_num, obj in lines:
            if id_ == dup_id:
                print(f"  📄 Line {line_num}: {json.dumps(obj, ensure_ascii=False)}")


❌ Duplicate IDs found:

🔁 ID 'tc_3' appears 2 times. Sample lines:
  📄 Line 2: {"id": "tc_3", "question": "Where in England was Dame Judi Dench born?", "answer": {"aliases": ["Park Grove (1895)", "York UA", "Yorkish", "UN/LOCODE:GBYRK", "York, UK", "Eoforwic", "Park Grove School", "York Ham", "The weather in York", "City of York", "York, England", "York, Yorkshire", "York ham", "County Borough of York", "YORK", "Eoferwic", "Park Grove Primary School", "York, North Yorkshire", "Yoisk", "York", "York (England)"], "normalized_aliases": ["york yorkshire", "eoferwic", "park grove primary school", "park grove school", "weather in york", "park grove 1895", "eoforwic", "county borough of york", "york uk", "un locode gbyrk", "city of york", "york england", "york ua", "york ham", "york", "yorkish", "yoisk", "york north yorkshire"], "matched_wiki_entity_name": "", "normalized_matched_wiki_entity_name": "", "normalized_value": "york", "type": "WikipediaEntity", "value": "York"}, "is_validation": f

In [12]:
import json
from collections import OrderedDict
from pathlib import Path

def deduplicate_jsonl_by_id(input_path, output_path=None, id_key="id"):
    """
    Deduplicate a JSONL file by a specified ID key.
    Keeps only the first occurrence of each ID.

    Args:
        input_path (str or Path): Path to the input JSONL file.
        output_path (str or Path, optional): Path to save the deduplicated file.
                                             If None, overwrites the original file.
        id_key (str): The key in each JSON object to use for deduplication.
    """
    input_path = Path(input_path)
    output_path = Path(output_path) if output_path else input_path

    seen = set()
    deduped = []

    with input_path.open("r", encoding="utf-8") as infile:
        for line in infile:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                obj_id = obj.get(id_key)
                if obj_id is not None and obj_id not in seen:
                    seen.add(obj_id)
                    deduped.append(obj)
            except json.JSONDecodeError:
                print(f"⚠️ Skipping invalid JSON line: {line}")

    print(f"✅ Found {len(seen)} unique entries (out of {len(deduped)} total)")

    with output_path.open("w", encoding="utf-8") as outfile:
        for obj in deduped:
            outfile.write(json.dumps(obj, ensure_ascii=False) + "\n")


# === Example usage ===
if __name__ == "__main__":
    deduplicate_jsonl_by_id(
        input_path="/home/guyb/tuner_knowledge/src/results/meta-llama_Llama-3.2-3B_scores.jsonl"
    )


✅ Found 76499 unique entries (out of 76499 total)


In [14]:
import json
from pathlib import Path
from textwrap import shorten

def find_first_corrupted_line(jsonl_path: str | Path, preview: int = 250):
    """
    Walk through a JSONL file and report the first line that fails json.loads().
    
    Parameters
    ----------
    jsonl_path : str | Path
        Path to the JSON Lines file.
    preview : int
        How many characters of the offending line to show (for debugging).
    """
    jsonl_path = Path(jsonl_path)
    with jsonl_path.open("r", encoding="utf-8") as f:
        for lineno, raw in enumerate(f, 1):
            line = raw.rstrip("\n")
            if not line:
                continue
            try:
                json.loads(line)
            except json.JSONDecodeError as e:
                # Found the first broken line – report and stop
                print(f"\n❌  JSON decode error on line {lineno}: {e}")
                print("Preview of offending line:")
                print(shorten(line, width=preview, placeholder=" …"))
                return lineno
    print("✅  File is fully valid JSONL – no errors found.")
    return None

# ---- run it ----
bad_line = find_first_corrupted_line(
    "/home/guyb/tuner_knowledge/src/results/meta-llama_Llama-3.2-3B_scores.jsonl"
)


✅  File is fully valid JSONL – no errors found.
